# IDS Challenge

## 01 Optimierung: Maximale Abdeckung der QS-Rundgänge durch die neue Roboterflotte

Die Einführung der Laufroboter in der Wartungsabteilung war ein Erfolg.
Das Unternehmen hat daher bereits **drei Laufroboter** angeschafft, die nun auch die Qualitätssicherung (QS)
beim Einsammeln der Materialproben unterstützen sollen.
Jeder Roboter kann maximal **5 Stunden am Stück** arbeiten, bevor seine Batterie aufgeladen werden muss.

Bisher haben drei Mitarbeiter*innen der Frühschicht (Gianni, Lissi und Ben) selbst entschieden, welche Maschinen
sie jeweils abgehen, um Materialproben einzusammeln. Insgesamt müssen so viele Maschinen besucht werden, dass selbst
drei Roboter mit ihrer begrenzten Batterielaufzeit **nicht alle davon innerhalb einer Schicht schaffen können**.
Die Geschäftsführung möchte daher Folgendes von Ihnen wissen:

1. **Wie viele der Maschinen können die drei Roboter zusammen maximal abdecken**, wenn jeder Roboter höchstens
   eine Rundtour von max. 5 Stunden fährt? Welche drei Rundtouren (Sequenzen) erreichen diese maximale Abdeckung?
2. Welche Maschinen bleiben übrig und müssen weiterhin von den Mitarbeiter*innen zu Fuß besucht werden? Jede übrig
   gebliebene Maschine wird dabei einzeln vom Depot aus angefahren und wieder zurückgegangen (keine gemeinsame
   Rundtour über mehrere übrig gebliebene Maschinen). Die **Summe dieser einzelnen Hin- und Rückwege** (Depot ->
   Maschine -> Depot) über alle übrig gebliebenen Maschinen soll möglichst klein sein. Es zählt also nicht nur,
   *wie viele* Maschinen die Roboter übernehmen, sondern auch, *welche* - eine Auswahl, die viele aber weit vom
   Depot entfernte Maschinen übrig lässt, kann für das verbleibende Team schlechter sein als eine Auswahl mit
   etwas weniger, aber näher am Depot liegenden übrig gebliebenen Maschinen.

Es handelt sich also um zwei miteinander verknüpfte Ziele: Maximieren Sie zunächst die Anzahl der von den Robotern
abgedeckten Maschinen, und versuchen Sie unter den dafür möglichen Auswahlen/Touren diejenige zu finden, die die
Summe der Depot-Distanzen der übrig gebliebenen Maschinen möglichst klein hält. Beschreiben Sie auch, wie Sie mit
diesem Zielkonflikt umgegangen sind.
Eine Rundtour beginnt und endet dabei immer am Labor der Qualitätssicherung (auch Depot genannt).

Dazu sind folgende Daten gegeben:
* Set an Maschinen mit Standorten, von denen Materialproben abgeholt werden müssen.
* Die drei Sets an Maschinen, die bisher von den drei Personen innerhalb der Frühschicht abgelaufen wurden.
  (Die Sequenz, die die Mitarbeiter*innen wählen ist unterschiedlich und leider unbekannt.)
* Eine Laufzeit-Servicezeit-Matrix, die die Laufzeiten zwischen den Maschinen und Servicezeiten
  (d.h. die Zeit, die der Roboterarm an der Maschine benötigt, um die Probe in der Kiste auf dem
  Rücken des Laufroboters zu platzieren) an den Maschinen enthält.

Joshi, aus dem Vorgängerteam, hat bereits einige Vorarbeiten geleistet, die Ihnen helfen sollen, die Aufgabe zu lösen.
Die Vorarbeiten sind in diesem Jupyter-Notebook enthalten.

Joshi hat bereits eine Funktion implementiert, die die Laufzeit einer Sequenz berechnet. Die Funktion `calc_total_tour_time()` finden Sie weiter unten.

In diesem Dokument sind die Vorarbeiten und eine kleine Nachricht vom Joshi enthalten!

## Unsere Vorarbeiten

Es existieren bereits einige Dateien, die wir vorbereitet haben. Sie liegen alle im Unterordner ``data/``.

### Maschinenliste für die drei Mitarbeiter*innen aus dem QS Team: gianni, lissi und ben
Die Dateien ``data/points_gianni_55_42.txt``, ``data/points_lissi_55_42.txt`` und ``data/points_ben_55_42.txt`` enthalten alle Maschinen, die die drei jeweils besucht haben. 

Die Inhalte der Dateien sehen wie folgt aus:
```` 
0; 0; 450.0; 0
43; 25.696035620363634; 437.3914169849524; 0
44; 293.39534810320026; 479.4355525765641; 1
...
````

Die erste Zahl ist der Index der Maschine oder auch der Index des Punktes, die zweite Zahl ist die x Koordinate, die zweite Zahl die y Koordinate und die dritte Zahl ist das Stockwerk. 
Der Start- und Endpunkt jeder Roboterrunde ist das Labor der Qualitätssicherung. Dieser Start-/Endpunkt heißt Depot und hat in allen drei Dateien den Index 0 (und steht in der ersten Zeile).


### Laufzeit-Servicezeit-Matrix

Zur Berechnung der gesamten Tourdauer benötigen wir neben der Toursequenz eine Laufzeit-Servicezeit-Matrix. 
Die LS Matrix haben wir im Textfile ``data/ls_matrix_55_42.txt`` abgespeichert.

Das File sieht wie folgt aus:

```
0; 0; 0
0; 1; 1000.0
0; 2; 1000.0
0; 3; 1123.606797749979
...
```

Jede Zeile steht für eine Verbindung (Kante) zwischen zwei Punkten (Maschinen oder Depot).
Die erste Zahl ist der Index des Startpunktes der Verbindung, die zweite Zahl der Index des Zielpunktes der Verbindung und die dritte Zahl 
ist die Laufzeit inkl. der Servicezeit an der Maschine, die der Endpunkt der Verbindung ist, in Sekunden.
Ist der Endpunkt einer Verbindung das Depot, dann wird natürlich keine Servicezeit benötigt.


### Tourdauer abhängig von einer Sequenz berechnen

Wir haben bereits eine Funktion implementiert, die die Tourdauer für eine gegebene Sequenz berechnet.
Dafür haben wir uns eine kleine Beispielmatrix mit nur 4 Maschinen und dem Start-/Endpunkt überlegt.
Die Distanzmatrix haben mit Hilfe eines Dictionary angegeben.
Eine Funktion, die die Textfiles einliest, haben wir leider noch nicht implementiert...

Ihr könnt also die Funktion ``calc_total_tour_time`` verwenden, um die Tourdauer für eine gegebene Sequenz zu berechnen.

In [3]:
def calc_total_tour_time(sequ, matrix):
    total_time = 0
    for i in range(len(sequ)-1):
        total_time += matrix[sequ[i], sequ[i+1]]
    return total_time

Hier der Test der Funktion mit unserer kleinen Distanzmatrix:

In [5]:
lsm_dict = {(0, 0): 0, (0, 1): 1000.0, (0, 2): 1000.0, (0, 3): 1123.606797749979, (0, 4): 1123.606797749979, (1, 0): 1000.0, (1, 1): 0, (1, 2): 1100.0, (1, 3): 1182.842712474619, (1, 4): 1100.0, (2, 0): 1000.0, (2, 1): 1100.0, (2, 2): 0, (2, 3): 1100.0, (2, 4): 1182.842712474619, (3, 0): 1123.606797749979, (3, 1): 1182.842712474619, (3, 2): 1100.0, (3, 3): 0, (3, 4): 1100.0, (4, 0): 1123.606797749979, (4, 1): 1100.0, (4, 2): 1182.842712474619, (4, 3): 1100.0, (4, 4): 0}

sequence = [0, 2, 3, 4, 1, 0]


tour_time_sec = calc_total_tour_time(sequence, lsm_dict)

print(f"Total time for the tour: {tour_time_sec} seconds")
print(f"Total time for the tour: {tour_time_sec/60} minutes")
print(f"Total time for the tour: {tour_time_sec/3600} hours")

Total time for the tour: 5300.0 seconds
Total time for the tour: 88.33333333333333 minutes
Total time for the tour: 1.4722222222222223 hours


# Unsere Vorüberlegungen

## Überblick verschaffen
Im ersten Schritt wollten wir uns mal einen Überblick verschaffen und alle Punkte aus der Liste von den drei Mitarbeiter*innen in eine Liste packen.
Da es sich ja um x-y-Koordinaten handelt, wollten wir uns einen 2D Scatterplot der Punkte machen und je nachdem in welchem Stockwerk sich die Maschine befindet, die Punkte unterschiedlich einfärben.
Außerdem wollten wir die Punkte mit den Indizes beschriften, damit wir später wissen, welche Maschine gemeint ist.
Wir könnten außerdem die Punkte mit unterschiedlichen Markierungen einzeichnen, je nachdem, ob es sich um Punkte von gianni, lissi oder ben handelt.
Vielleicht hilft Euch ein LLM beim Erstellen des Scatterplots.

## Algorithmus zur Bestimmung der Sequenz
Wir kennen uns nicht gut mit Optimierung aus.
Wir haben gehört, dass es Algorithmen gibt, die die optimale Sequenz (Sequenzen werden wohl oft auch Touren genannt), d.h. die kürzest mögliche Sequenz, bei gegebener LSM berechnen.
Aber wir würden erst mal einen einfachen Algorithmus implementieren, der eine Sequenz Schritt für Schritt aufbaut und immer den nächsten noch nicht besuchten Nachbarn anhängt.
Diesmal müssen wir aber zusätzlich nach jedem Schritt prüfen, ob die 5 Stunden noch reichen: Bevor wir eine weitere Maschine anhängen, schauen wir, ob die bisherige Tour plus der Weg zur nächsten Maschine plus der Rückweg zum Depot noch innerhalb der 5 Stunden bleibt.
Wenn nicht, brechen wir die Tour ab und schließen sie, indem wir zum Depot zurückkehren - das ist dann die fertige Tour für einen Roboter.
Das Ganze könnten wir für alle drei Roboter wiederholen - dann müssten wir aber aufpassen, dass nicht alle drei Roboter immer wieder bei den gleichen, naheliegendsten Maschinen anfangen.
Alternativ könntet ihr versuchen, etablierte Algorithmen zu finden (Achtung, die solltet Ihr dann aber verstehen und in der Präsentation erklären können).

Wir haben überlegt, dass vielleicht ein ganz einfacheres Setup zum Testen gut wäre.
Wir wollten uns eine kleine Distanzmatrix mit nur 3 Maschinen und dem Start-/Endpunkt überlegen.
Dazu würden wir einfach annehmen, dass wir im 2 dimensionalen Raum ohne Hindernisse sind und der Start-/Endpunkt an der Position (0,0) (Index 0) liegt und die Maschinen an den Punkten (10,10) (Index 1), (0,10) (Index 2) und (10,0) (Index 3) sind.
Damit könnten wir einfach eine Laufzeitmatrix über die Euklidische Distanz berechnen (es gibt ja keine Wege oder Hindernisse) und wir können uns das ganz einfach aufmalen.
Wir tun einfach so, als würde eine Entfernungseinheit in einer Sekunde zurückgelegt werden.
Die kürzeste Sequenz wäre dann einfach die Sequenz ``[0, 2, 1, 3, 0]`` oder auch umgekehrt ``[0, 3, 1, 2, 0]``.

Solche ganz einfachen Beispiele helfen uns oft, um zu verstehen, wie wir die Algorithmen implementieren müssen.
Wenn es für das einfache Beispiel funktioniert, dann können wir uns ein etwas komplexeres ausdenken und dann auf die echten Daten anwenden.

Vielleicht findet Ihr einen Plotvariant, um die Lösung im Scatterplot zu visualisieren.

Zur Erinnerung die Formel für Euklidische Distanz zwischen zwei Punkten: $\sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$

## Aufteilung auf die drei Roboter
Wir haben außerdem beim Beobachten der QS Mitarbeiter*innen gesehen, dass die Zuteilung von Maschinen zu Mitarbeiter*innen nicht besonders gut ist - vielleicht seht Ihr das auch in Eurem Scatterplot.
Da könnte man die Maschinen ja mal in drei räumlich sinnvolle Gruppen aufteilen (eine Gruppe pro Roboter) und dann pro Gruppe eine eigene Tour bestimmen.
Vermutlich passt nicht jede Gruppe komplett in die 5 Stunden eines Roboters - dann bleiben in der jeweiligen Gruppe ein paar Maschinen übrig.
Für die übrig gebliebenen Maschinen aus allen drei Gruppen zählt am Ende nur die Summe der einzelnen Hin- und Rückwege vom Depot - wie Ihr diese Summe möglichst klein haltet, ist Teil der Aufgabe.

Das sind unsere Ideen, aber vielleicht habt Ihr ganz andere Ideen und findet bessere Ansätze und Algorithmen!

Viel Erfolg! Euer Joshi!

### Abnahmekriterien

* Jupyter-Notebook `abgabe_tp01_optimierung.ipynb` mit Darstellung der groben Idee und mit der Implementierung eines oder mehrerer Algorithmen
* Verwendung von Funktionen, um gut lesbaren Code zu schreiben (Tipp: eine Funktion sollte nicht mehr als 10 Zeilen Code enthalten. Enthält sie mehr, sollte sie in kleinere Funktionen aufgeteilt werden)
* Einschätzung zur Laufzeitkomplexität der Algorithmen (n = Anzahl Maschinen)
* Drei Rundtouren für die Roboter, die jeweils die 5-Stunden-Grenze einhalten, und eine Angabe, wie viele Maschinen damit insgesamt abgedeckt werden
* Liste der übrig gebliebenen Maschinen sowie die Summe der einzelnen Hin- und Rückwege vom Depot zu diesen Maschinen, möglichst klein gehalten
* Nachvollziehbare Beschreibung, wie mit dem Zielkonflikt zwischen maximaler Abdeckung und kleiner Distanzsumme der übrig gebliebenen Maschinen umgegangen wurde
* Ansprechende Präsentation der Algorithmen und Ergebnisse im Vortrag (insgesamt 10 Minuten)